In [2]:
import os
import json
from dotenv import load_dotenv
from langchain_openai import ChatOpenAI

from urbanomy.methods.land_value_modeling import (
    build_pareto_front_dataframe,
    run_pareto_vote,
)

load_dotenv()

llm = ChatOpenAI(
    model=os.getenv("CHAT_MODEL"),
    base_url=os.getenv("BASE_URL") + "/v1",
    temperature=float(os.getenv("TEMPERATURE", "0")),
    api_key=os.getenv("OPENAI_API_KEY") or "dummy",
)



In [3]:
import pandas as pd

pareto_front_df = pd.read_pickle("pareto_front_df.pkl")
pareto_front_df

,scenario_id,title,summary,land_use,land_value_after,land_value_gain,investor_npv,params_repaired
0,scenario_0,scenario_0 | RESIDENTIAL,scenario_0 | RESIDENTIAL; land_use=RESIDENTIAL...,RESIDENTIAL,6.762501e+10,1.320421e+09,5.427989e+09,"{'footprint_area': 25830.515598412425, 'l': 8...."
1,scenario_1,scenario_1 | RESIDENTIAL,scenario_1 | RESIDENTIAL; land_use=RESIDENTIAL...,RESIDENTIAL,6.759989e+10,1.295302e+09,5.893216e+09,"{'footprint_area': 25390.982187701644, 'l': 9...."
2,scenario_2,scenario_2 | RESIDENTIAL,scenario_2 | RESIDENTIAL; land_use=RESIDENTIAL...,RESIDENTIAL,6.762964e+10,1.325053e+09,5.263091e+09,"{'footprint_area': 25830.515598412425, 'l': 8...."
3,scenario_3,scenario_3 | RESIDENTIAL,scenario_3 | RESIDENTIAL; land_use=RESIDENTIAL...,RESIDENTIAL,6.752879e+10,1.224198e+09,6.040916e+09,"{'footprint_area': 25831.604654000806, 'l': 9...."


In [4]:
result = run_pareto_vote(
    pareto_front_df,
    llm=llm,
    additional_context=(
        "Нужно выбрать лучший сценарий из парето-фронта. "
        "Важно учитывать баланс городской выгоды, качества жизни и доходности."
    ),
    scenario_id_column="scenario_id",
    title_column="title",
    summary_column="summary",
)


Лучший сценарий
 • ID: scenario_1
 • Название: scenario_1 | RESIDENTIAL
 • Summary: scenario_1 | RESIDENTIAL; land_use=RESIDENTIAL; прирост стоимости земли=1 295 301 611 ₽; NPV инвестора=5 893 215 666 ₽
 • Метрики: land_value_after=67599891421.54619, land_value_gain=1295301611.0852966, investor_npv=5893215666.1

Почему выбран
 • Сценарий scenario_1 выбран, потому что он лучше балансирует интересы сторон: прирост стоимости земли 1 295 301 611 ₽, NPV инвестора 5 893 215 666 ₽, доминирующий land_use=RESIDENTIAL. Поддержавшие стороны объяснили выбор так: Администрация: Сценарий 1 обеспечивает наилучший баланс общественной пользы и финансовой устойчивости: высокий прирост стоимости земли (≈1,30 млрд ₽) усиливает налоговую базу, а NPV инвестора (~5,89 млрд ₽) гарантирует приток доходов. Кроме того, в нём максимальное население (~12 000 человек) и высокий коэффициент застройки (FSI ≈ 0,92), что повышает рабочие места и эффективность использования земли, при умеренных транспортных нагрузках. Д

In [9]:
from urbanomy.methods.land_value_modeling import (
    run_pareto_vote,
    ask_winner_scenario_question,
)

qa_result = ask_winner_scenario_question(
    result=result,
    llm=llm,
    question="Сколько прибыли он принесет если увеличить этажность на 1?",
)

Ответ по выбранному сценарию
 • Вопрос: Сколько прибыли он принесет если увеличить этажность на 1?
 • Ответ: Недостаточно данных для расчёта изменения прибыли при увеличении этажности на один этаж в выбранном сценарии.

Ключевые пункты
 • Имеются только итоговые показатели NPV инвестора и коэффициент застройки (FSI) для текущего сценария
 • Нет информации о том, как изменение этажности влияет на доходы, расходы или NPV
 • Отсутствуют параметры стоимости строительства за дополнительный этаж и его влияние на доходность

Ограничения
 • Отсутствуют данные о стоимости и доходности дополнительного этажа
 • Не известна зависимость NPV инвестора от изменения FSI или количества этажей
 • Нет модели, позволяющей экстраполировать прибыль при изменении этажности
